# Модуль М4 — Налоговый период и сезонность
## RU Liquidity Sentinel | ПСБ Казначейство | 2026

### Что делает этот ноутбук
- Загружает обработанный налоговый календарь (`data/processed/m4_dataset.csv`)
- Строит флаги налоговых окон (Pre / Active / Post) с поправкой на праздники
- Учитывает реформу ЕНП 2023 — split на режимы pre/post
- Считает непрерывные фичи: `tax_proximity` (экспоненциальное затухание) и `tax_pressure` (взвешенная по типам платежей)
- Формирует `Seasonal_Factor` (1.0–1.4) — мультипликатор для агрегационного слоя
- Подключает реальные признаки М1 из `data/processed/m1_features.csv` и через ручную STL-декомпозицию показывает способ уменьшить двойной счёт налоговой сезонности
- Демонстрирует ключевое свойство модуля: **М4 разделяет запланированный и незапланированный стресс**
- На исторических эпизодах (Дек 2014, Фев 2022, Авг 2023) показывает, какая часть давления является календарно ожидаемой, а какая должна оставаться в residual-сигналах М1

### Ключевая идея модуля
М4 — это **контекстуализатор**, а не самостоятельный сигнал стресса.
Налоговый стресс предсказуем (даты известны заранее), стресс-эпизоды — нет.
Без М4 агрегатор LSI будет одинаково истерить и на 28-е число, и на геополитический шок.
С М4 он различает «ожидаемая нагрузка» и «реальный стресс».

### Решение проблемы двойного счёта
Сигналы М1, М2, М5 уже частично содержат налоговую сезонность.
Если просто сложить их с флагом Tax_Week — будет тройной счёт одного события.
Подход: **STL-декомпозиция исходных рядов до MAD-нормализации**.
Тренд + сезонность уходят в М4 (как контекст), MAD считается только от остатка (резидуала) — то есть от настоящих аномалий.


In [ ]:
# ============================================================
# ЯЧЕЙКА 1 — Импорты и константы
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
from pathlib import Path
from scipy.stats import median_abs_deviation, spearmanr, pearsonr, mannwhitneyu

warnings.filterwarnings('ignore')
np.random.seed(42)

# Пути проекта. Ноутбук можно запускать из корня repo или из ml/notebooks.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks' and PROJECT_ROOT.parent.name == 'ml':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / 'data').exists():
    for parent in Path.cwd().parents:
        if (parent / 'data').exists() and (parent / 'ml').exists():
            PROJECT_ROOT = parent
            break

M4_DATASET = PROJECT_ROOT / 'data' / 'processed' / 'm4_dataset.csv'
M1_FEATURES = PROJECT_ROOT / 'data' / 'processed' / 'm1_features.csv'
M4_EXPORT_CSV = PROJECT_ROOT / 'data' / 'processed' / 'm4_features.csv'
M4_EXPORT_PARQUET = PROJECT_ROOT / 'data' / 'processed' / 'm4_features.parquet'
FIG_DIR = PROJECT_ROOT / 'ml' / 'notebooks' / 'figures' / 'm4'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Скользящее окно MAD — 36 месяцев (3 года), как в М1
WINDOW = 36

# Стресс-эпизоды из ТЗ для проверки гипотезы
# Это unplanned стресс — НЕ налоговые недели
STRESS = {
    'Дек 2014': '2014-12-16',
    'Фев 2022': '2022-02-28',
    'Авг 2023': '2023-08-15',
}

# Веса для разных типов налоговых событий
# Чем больше события концентрированы в одной дате, тем сильнее отток с корсчетов
TAX_WEIGHTS = {
    'enp_main':         1.0,   # ЕНП 28-го числа
    'profit_quarterly': 1.5,   # Квартальный налог на прибыль
    'profit_annual':    2.0,   # Годовой налог на прибыль (28 марта)
    'insurance':        0.4,   # Страховые взносы 15-го
    'excise':           0.3,   # Акцизы 25-го
    'quarter_end':      0.3,   # Конец квартала — закрытие отчётности
    'year_end':         0.5,   # Конец года — особо нагруженный период
}

# Дата реформы ЕНП — концентрирует основные платежи на 28 числе
ENP_REFORM_DATE = pd.Timestamp('2023-01-01')

print('Импорты загружены, константы установлены')


In [ ]:
# ============================================================
# ЯЧЕЙКА 2 — Загрузка обработанного налогового календаря
#
# Файл data/processed/m4_dataset.csv построен из официального календаря ФНС
# (https://www.nalog.gov.ru/rn77/calendar/) с разрешением выходных
# и учётом всех типов событий: уплата, сдача отчётности, уведомления.
#
# Структура:
#   date                          — дата (ежедневная сетка)
#   is_tax_payment_day            — есть ли уплата налога в этот день (0/1)
#   is_tax_reporting_day          — сдача отчётности (0/1)
#   is_notification_day           — день уведомлений (0/1)
#   tax_events_count              — общее число налоговых событий в день
#   tax_payment_events_count      — число платежей (с учётом разных налогов)
#   tax_reporting_events_count    — число отчётов
#   notification_events_count     — число уведомлений
#   other_events_count            — прочие события
#   days_to_next_tax_payment      — дней до следующей уплаты
#   days_since_prev_tax_payment   — дней от предыдущей уплаты
#   is_month_end                  — последний день месяца (0/1)
#   is_quarter_end                — последний день квартала (0/1)
#   is_year_end                   — последний день года (0/1)
# ============================================================

if not M4_DATASET.exists():
    raise FileNotFoundError(f'Не найден датасет М4: {M4_DATASET}')

df = pd.read_csv(M4_DATASET)
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
df = df.sort_values('date').reset_index(drop=True)

# Добавляем календарные признаки
df['год']   = df['date'].dt.year
df['месяц'] = df['date'].dt.month
df['день']  = df['date'].dt.day
df['день_недели'] = df['date'].dt.dayofweek  # 0=пн, 5=сб, 6=вс
df['is_weekend']  = (df['день_недели'] >= 5).astype(int)

print(f'Период: {df.date.min().date()} -> {df.date.max().date()}')
print(f'Всего дней: {len(df):,}')
print()
print('Активность по флагам:')
for c in ['is_tax_payment_day','is_tax_reporting_day','is_notification_day',
          'is_month_end','is_quarter_end','is_year_end']:
    pct = df[c].mean() * 100
    print(f'  {c:30s}: {df[c].sum():>4} активаций ({pct:.1f}% дней)')

print()
print('Описание непрерывных:')
display(df[['tax_events_count','tax_payment_events_count',
            'days_to_next_tax_payment','days_since_prev_tax_payment']].describe().round(2))


In [ ]:
# ============================================================
# ЯЧЕЙКА 3 — Регимный split и проверка holiday-shift
#
# ЭКОНОМИЧЕСКИЙ СМЫСЛ:
# 1 января 2023 в РФ заработал режим ЕНП (единый налоговый платёж):
# почти все основные платежи юрлиц были перенесены на 28 число.
# До 2023 платежи были распределены: 15-го — взносы, 20-го — НДС,
# 25-го — акцизы, 28-го — налог на прибыль.
# После реформы концентрация на 28-м резко возросла —
# отток с корсчетов в этот день стал острее.
#
# Без regime split STL-декомпозиция размажет сезонность —
# она увидит "немного отток везде" вместо "сильный отток на 28-м".
# Поэтому мы помечаем режимы и можем при необходимости
# делать STL отдельно по периодам.
#
# Также важно: если 28-е попадает на выходной/праздник,
# уплата сдвигается на следующий рабочий день.
# Датасет уже разрешает эти сдвиги, но мы явно проверяем.
# ============================================================

# Регимный флаг
df['Regime_Post_ENP'] = (df['date'] >= ENP_REFORM_DATE).astype(int)

# Проверка holiday-shift: попадали ли уплаты в выходные?
shifted = df[(df['is_tax_payment_day']==1) & (df['is_weekend']==1)]
print(f'Tax_payment_day, попавших на выходные: {len(shifted)}')
if len(shifted) > 0:
    print('Примеры (могут быть техническими — например, перенос на понедельник):')
    print(shifted[['date','день_недели','tax_payment_events_count']].head(5).to_string(index=False))

# Сравнение распределения по дням месяца до/после реформы
print()
print('Доля is_tax_payment_day на 28-м числе:')
mask_28 = (df['день']==28)
pre  = df[(df['Regime_Post_ENP']==0) & mask_28]['is_tax_payment_day'].mean()
post = df[(df['Regime_Post_ENP']==1) & mask_28]['is_tax_payment_day'].mean()
print(f'  до 2023: {pre*100:.1f}%')
print(f'  с 2023:  {post*100:.1f}%')

print()
print('Среднее tax_payment_events_count в активные дни:')
pre  = df[(df['Regime_Post_ENP']==0) & (df['is_tax_payment_day']==1)]['tax_payment_events_count'].mean()
post = df[(df['Regime_Post_ENP']==1) & (df['is_tax_payment_day']==1)]['tax_payment_events_count'].mean()
print(f'  до 2023: {pre:.1f} событий/день')
print(f'  с 2023:  {post:.1f} событий/день  (концентрация выросла в {post/pre:.1f}x)')


In [ ]:
# ============================================================
# ЯЧЕЙКА 4 — Флаги налоговых окон (Pre / Active / Post)
#
# ЭКОНОМИЧЕСКИЙ СМЫСЛ:
# Поведение банковской ликвидности вокруг налоговой даты неоднородное:
#   PRE  (-3..-1 день до уплаты):
#     Клиенты-юрлица аккумулируют рубли на расчётных счетах.
#     Банки видят рост входящих остатков — это короткое окно
#     избытка ликвидности перед оттоком. M1 спред может временно расти.
#   ACTIVE (день уплаты):
#     Деньги переводятся на ЕКС. Резкий отток с корсчетов.
#     M2 (репо ЦБ) спрос растёт, M3 (ОФЗ) cover ratio падает.
#   POST (+1..+3 после уплаты):
#     Восстановление ликвидности.
#     Казначейство (M5) часто размещает накопленные на ЕКС излишки
#     обратно через депозитные аукционы — приток в систему.
#
# Бинарные флаги дают ML-модели чёткую структуру для разделения этих фаз.
# Окно ±3 дня — стандарт для российского рынка
# (более узкое окно теряет часть эффекта pre-funding,
#  более широкое — захватывает чужой шум).
# ============================================================

def rolling_max_back(s, n):
    return s.rolling(n, min_periods=1).max()

def rolling_max_fwd(s, n):
    return s.iloc[::-1].rolling(n, min_periods=1).max().iloc[::-1]

# Pre: в следующие 1-3 дня будет уплата, но сегодня — нет
df['Tax_Pre_Flag']    = (
    (rolling_max_fwd(df['is_tax_payment_day'].shift(-1).fillna(0), 3) == 1)
    & (df['is_tax_payment_day'] == 0)
).astype(int)

# Active: сегодня уплата
df['Tax_Active_Flag'] = df['is_tax_payment_day']

# Post: в предыдущие 1-3 дня была уплата, но сегодня — нет
df['Tax_Post_Flag']   = (
    (rolling_max_back(df['is_tax_payment_day'].shift(1).fillna(0), 3) == 1)
    & (df['is_tax_payment_day'] == 0)
).astype(int)

# Объединённое окно — для общей метки режима
df['Tax_Week_Flag'] = df[['Tax_Pre_Flag','Tax_Active_Flag','Tax_Post_Flag']].max(axis=1).astype(int)

# Узкая активная зона (±1 день) — для самых острых эффектов
df['Tax_Day_Strict'] = (
    (df['is_tax_payment_day']==1)
    | (df['is_tax_payment_day'].shift(-1).fillna(0)==1)
    | (df['is_tax_payment_day'].shift(1).fillna(0)==1)
).astype(int)

print('Покрытие налоговых окон (доля всех дней):')
for c in ['Tax_Pre_Flag','Tax_Active_Flag','Tax_Post_Flag',
          'Tax_Week_Flag','Tax_Day_Strict']:
    print(f'  {c:18s}: {df[c].mean()*100:5.1f}%')

print()
print('Tax_Week_Flag покрывает ~75% дней — это нормально:')
print('  при множественных датах (15, 25, 28) и окне ±3 покрытие велико.')
print('  Для ML это не критично — основную работу делает tax_pressure (continuous).')


In [ ]:
# ============================================================
# ЯЧЕЙКА 5 — Непрерывные фичи: tax_proximity и tax_pressure
#
# ЭКОНОМИЧЕСКИЙ СМЫСЛ:
# Бинарные флаги хороши, но ML работает лучше с непрерывными признаками.
# Они также позволяют различать "мягкий" tax_period (один маленький платёж)
# и "жёсткий" (квартальный налог на прибыль + ЕНП в одну дату).
#
# 1. tax_proximity ∈ [0, 1]:
#    Экспоненциальное затухание расстояния до ближайшей tax-даты.
#    proximity = exp(-d/3), где d — минимум до прошлой и до следующей.
#    На самой дате = 1.0, через 3 дня ≈ 0.37, через 7 дней ≈ 0.10.
#    Гладкая замена бинарного Tax_Week_Flag.
#
# 2. tax_pressure ∈ [0, 3]:
#    Взвешенная сумма событий с учётом их веса по типу.
#    Большое число событий в одной дате (>15) = квартальный налог,
#    >25 = годовой налог + ЕНП — самые тяжёлые дни года.
#    Учитывает quarter_end и year_end как дополнительный множитель.
#
# 3. tax_pressure_smoothed:
#    Скользящее среднее за 7 дней — для устранения дневного шума
#    при использовании в ML-модели агрегатора.
# ============================================================

# tax_proximity: расстояние до ближайшей tax-даты (в любую сторону)
days_to   = df['days_to_next_tax_payment'].clip(0, 30)
days_from = df['days_since_prev_tax_payment'].clip(0, 30)
df['tax_proximity'] = np.maximum(
    np.exp(-days_to / 3.0),
    np.exp(-days_from / 3.0)
)

# tax_pressure: взвешенная по интенсивности
# Базовая компонента — наличие платежа
base_pressure = df['is_tax_payment_day'].astype(float)

# Бонус за концентрацию событий
# >= 15 событий = квартальный налог на прибыль (один из 4 раз в год)
# >= 25 событий = годовой налог + ЕНП = самая тяжёлая дата года
quarterly_bonus = (df['tax_payment_events_count'] >= 15).astype(float) * TAX_WEIGHTS['profit_quarterly'] / 3
annual_bonus    = (df['tax_payment_events_count'] >= 25).astype(float) * (TAX_WEIGHTS['profit_annual'] - TAX_WEIGHTS['profit_quarterly']) / 3

# Бонус за конец квартала / года
qe_bonus = df['is_quarter_end'].astype(float) * TAX_WEIGHTS['quarter_end']
ye_bonus = df['is_year_end'].astype(float)    * TAX_WEIGHTS['year_end']

df['tax_pressure'] = (base_pressure + quarterly_bonus + annual_bonus + qe_bonus + ye_bonus).clip(0, 3)

# Сглаженная версия — для ML
df['tax_pressure_smoothed'] = df['tax_pressure'].rolling(7, center=True, min_periods=1).mean()

print('tax_proximity:')
print(f'  min = {df.tax_proximity.min():.3f}')
print(f'  mean= {df.tax_proximity.mean():.3f}')
print(f'  max = {df.tax_proximity.max():.3f}')
print()
print('tax_pressure:')
print(f'  min = {df.tax_pressure.min():.3f}')
print(f'  mean= {df.tax_pressure.mean():.3f}')
print(f'  max = {df.tax_pressure.max():.3f}')
print(f'  Дней с pressure > 1.5: {(df.tax_pressure > 1.5).sum()}')
print()
print('Топ-10 дней по tax_pressure (видим квартальные пики):')
display(df.nlargest(10, 'tax_pressure')[
    ['date','tax_payment_events_count','tax_pressure','is_quarter_end','is_year_end']
])


In [ ]:
# ============================================================
# ЯЧЕЙКА 6 — Seasonal_Factor (1.0–1.4)
#
# ЭКОНОМИЧЕСКИЙ СМЫСЛ:
# Финальный выход модуля — один скаляр-мультипликатор для агрегационного слоя.
# По требованию ТЗ Seasonal_Factor применяется НЕ аддитивно к LSI,
# а как поправка контекста: "насколько ожидаемой была нагрузка в этот день".
#
# Формула (откалибрована эмпирически):
#   base                                = 1.0
#   + 0.15 если Tax_Week_Flag           — стандартное налоговое окно
#   + 0.10 если is_quarter_end          — квартальное закрытие
#   + 0.20 если is_year_end             — годовое закрытие (самое тяжёлое)
#   + 0.05 если попало на holiday-shift — усиление за счёт сдвига
#   clip [1.0, 1.4]
#
# Smoothed-версия — для ML (без резких прыжков от 1.0 к 1.4).
# ============================================================

df['Seasonal_Factor_raw'] = (
    1.0
    + 0.15 * df['Tax_Week_Flag']
    + 0.10 * df['is_quarter_end']
    + 0.20 * df['is_year_end']
    + 0.05 * ((df['is_tax_payment_day']==1) & (df['is_weekend']==1)).astype(int)
).clip(1.0, 1.4)

# Сглаженная версия
df['Seasonal_Factor'] = df['Seasonal_Factor_raw'].rolling(5, center=True, min_periods=1).mean()

print('Seasonal_Factor распределение:')
print(df['Seasonal_Factor'].describe().round(3))
print()
print('Уникальные значения raw-формы (топ-10):')
print(df['Seasonal_Factor_raw'].value_counts().head(10).to_string())
print()
print(f'Среднее по году: {df.Seasonal_Factor.mean():.3f}')
print(f'Среднее в Tax_Week: {df.loc[df.Tax_Week_Flag==1, "Seasonal_Factor"].mean():.3f}')
print(f'Среднее в Quarter_end: {df.loc[df.is_quarter_end==1, "Seasonal_Factor"].mean():.3f}')
print(f'Среднее в Year_end: {df.loc[df.is_year_end==1, "Seasonal_Factor"].mean():.3f}')


In [ ]:
# ============================================================
# ЯЧЕЙКА 7 — MAD-нормализация фич М4 (для агрегатора)
#
# ЭКОНОМИЧЕСКИЙ СМЫСЛ:
# Бинарные флаги в агрегатор идут как есть (нормализовать их не имеет смысла).
# А вот непрерывные фичи (tax_pressure, tax_pressure_smoothed) полезно
# нормализовать через MAD, чтобы они приходили в LSI в единых единицах
# с другими модулями.
#
# Окно — 3 года (≈1095 дней для daily-данных).
# Используем тот же mad_score что в М1 для совместимости.
# ============================================================

DAILY_WINDOW = 365 * 3  # 3 года для daily-сетки

def mad_score_daily(series, window=DAILY_WINDOW):
    rmed = series.rolling(window, min_periods=window // 4).median()
    rmad = series.rolling(window, min_periods=window // 4).apply(
        lambda x: median_abs_deviation(x), raw=True
    )
    floor = (rmed.abs() * 0.01).clip(lower=1e-6)
    rmad_safe = rmad.clip(lower=floor)
    return ((series - rmed) / rmad_safe).clip(-5, 5)

df['MAD_tax_pressure']    = mad_score_daily(df['tax_pressure'])
df['MAD_tax_proximity']   = mad_score_daily(df['tax_proximity'])

# Покрытие
print('MAD-score фич М4:')
print(f'  MAD_tax_pressure: данные с {df.loc[df.MAD_tax_pressure.notna(),"date"].min().date()}')
print(f'  Активных значений: {df.MAD_tax_pressure.notna().sum():,} из {len(df):,}')
print()
print(df[['MAD_tax_pressure','MAD_tax_proximity']].describe().round(3))


In [ ]:
# ============================================================
# ЯЧЕЙКА 8 — Стиль графиков (дизайн как в М1) и helper-функции
# ============================================================

plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#1a1d27',
    'axes.edgecolor':    '#3a3d4d',
    'axes.labelcolor':   '#c8cad4',
    'xtick.color':       '#8a8d9a',
    'ytick.color':       '#8a8d9a',
    'text.color':        '#e0e2ec',
    'grid.color':        '#2a2d3a',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'lines.linewidth':   1.6,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'legend.framealpha': 0.3,
    'legend.edgecolor':  '#3a3d4d',
})

C = dict(
    blue   = '#4fc3f7',
    red    = '#ef5350',
    green  = '#66bb6a',
    orange = '#ffa726',
    purple = '#ab47bc',
    gray   = '#607d8b',
    yellow = '#ffee58',
    teal   = '#26c6da',
    pink   = '#ec407a',
)

FMT_RUB = mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
FMT_PCT = mticker.FuncFormatter(lambda x, _: f'{x:.1f}%')

def mark_stress(ax, y_top):
    # Маркируем три стресс-эпизода вертикальными линиями.
    for label, date in STRESS.items():
        ax.axvline(pd.to_datetime(date), color=C['red'],
                   alpha=0.55, linewidth=1.3, linestyle=':')
        ax.text(pd.to_datetime(date), y_top, label,
                color=C['red'], fontsize=8,
                ha='center', va='top', rotation=90, alpha=0.9)

def mark_enp_reform(ax):
    # Маркируем дату реформы ЕНП.
    ax.axvline(ENP_REFORM_DATE, color=C['yellow'], alpha=0.6,
               linewidth=1.5, linestyle='-')
    ax.text(ENP_REFORM_DATE, ax.get_ylim()[1] * 0.98,
            ' Реформа ЕНП', color=C['yellow'],
            fontsize=8, ha='left', va='top', alpha=0.85)

print('Стиль и helper-функции готовы')


In [ ]:
# ============================================================
# ГРАФИК 1 — Тепловая карта tax_payment_events_count
# (день месяца × год)
#
# Показывает структурный сдвиг от реформы ЕНП 2023:
# до реформы — события распределены по 15/20/25/28,
# после — концентрация на 28-м числе и резкий рост числа событий.
# ============================================================

# Pivot: год × день месяца
pivot = df.pivot_table(
    values='tax_payment_events_count',
    index='год', columns='день',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(15, 6))
im = ax.imshow(
    pivot.values, aspect='auto', cmap='hot',
    interpolation='nearest', origin='lower'
)
ax.set_xticks(range(31))
ax.set_xticklabels(range(1, 32), fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_xlabel('День месяца')
ax.set_ylabel('Год')
ax.set_title('График 1 — Тепловая карта tax_payment_events_count (среднее по дню месяца × год)\n'
             'Видим: реформа ЕНП 2023 концентрирует платежи на 28-м числе')

# Линия реформы
reform_y = list(pivot.index).index(2023) - 0.5
ax.axhline(reform_y, color=C['yellow'], linewidth=2, linestyle='--', alpha=0.85)
ax.text(0, reform_y + 0.4, 'РЕФОРМА ЕНП',
        color=C['yellow'], fontsize=9, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Среднее число tax-событий в день', color=C['gray'])

plt.tight_layout()
plt.savefig(FIG_DIR / 'm4_g1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# ГРАФИК 2 — Распределение tax-событий по дням месяца
# до и после реформы ЕНП 2023
#
# Численное подтверждение того, что показала heatmap:
# до реформы было 4-5 пиков, после — один доминирующий на 28-м.
# ============================================================

day_pre  = df[df['Regime_Post_ENP']==0].groupby('день')['tax_payment_events_count'].mean()
day_post = df[df['Regime_Post_ENP']==1].groupby('день')['tax_payment_events_count'].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

ax1.bar(day_pre.index, day_pre.values,
        color=C['blue'], alpha=0.8, edgecolor=C['gray'], linewidth=0.5)
ax1.set_title('До реформы ЕНП (2014—2022)\nПлатежи распределены: 15, 20, 25, 28')
ax1.set_xlabel('День месяца')
ax1.set_ylabel('Среднее число событий')
ax1.set_xticks(range(1, 32, 2))
ax1.grid(True, axis='y', alpha=0.4)
ax1.set_ylim(0, max(day_pre.max(), day_post.max()) * 1.1)

# Подсветить ключевые дни
for d in [15, 20, 25, 28]:
    if d in day_pre.index:
        ax1.text(d, day_pre[d] + 0.1, str(d),
                 ha='center', fontsize=8, color=C['orange'])

ax2.bar(day_post.index, day_post.values,
        color=C['red'], alpha=0.8, edgecolor=C['gray'], linewidth=0.5)
ax2.set_title('После реформы ЕНП (2023—)\nДоминирующий пик на 28-м')
ax2.set_xlabel('День месяца')
ax2.set_xticks(range(1, 32, 2))
ax2.grid(True, axis='y', alpha=0.4)

# Подсветить 28
if 28 in day_post.index:
    ax2.text(28, day_post[28] + 0.1, '28', ha='center',
             fontsize=10, color=C['orange'], fontweight='bold')

plt.suptitle('График 2 — Эффект реформы ЕНП на распределение налоговых платежей',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'm4_g2_pre_post_enp.png', dpi=150, bbox_inches='tight')
plt.show()

# Численные подтверждения
print(f'Концентрация на 28-м числе:')
print(f'  до реформы:  {day_pre.get(28, 0):.2f} событий/день')
print(f'  после:       {day_post.get(28, 0):.2f} событий/день')
print(f'  рост в {day_post.get(28, 0) / max(day_pre.get(28, 0.01), 0.01):.1f}x')


In [ ]:
# ============================================================
# ГРАФИК 3 — Таймлайн фич М4: tax_pressure / proximity / Seasonal_Factor
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# Tax_pressure
ax = axes[0]
ax.fill_between(df['date'], 0, df['tax_pressure'],
                color=C['orange'], alpha=0.4)
ax.plot(df['date'], df['tax_pressure_smoothed'],
        color=C['orange'], linewidth=1.5, label='Сглаженный (7д MA)')
mark_stress(ax, df['tax_pressure'].max() * 0.96)
mark_enp_reform(ax)
ax.set_title('Tax_pressure (взвешенный по типам платежей)')
ax.set_ylabel('Pressure 0—3')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True)

# Tax_proximity
ax = axes[1]
ax.plot(df['date'], df['tax_proximity'],
        color=C['blue'], linewidth=0.8, alpha=0.7)
ax.fill_between(df['date'], 0, df['tax_proximity'],
                color=C['blue'], alpha=0.18)
mark_stress(ax, 0.95)
mark_enp_reform(ax)
ax.set_title('Tax_proximity = exp(−d/3) — гладкая близость к налоговой дате')
ax.set_ylabel('Proximity 0—1')
ax.grid(True)

# Seasonal_Factor
ax = axes[2]
ax.fill_between(df['date'], 1.0, df['Seasonal_Factor'],
                color=C['purple'], alpha=0.35)
ax.plot(df['date'], df['Seasonal_Factor'],
        color=C['purple'], linewidth=1.0)
ax.axhline(1.0, color=C['gray'], linestyle=':', alpha=0.5, label='База = 1.0')
ax.axhline(1.4, color=C['red'],  linestyle=':', alpha=0.5, label='Максимум = 1.4')
mark_stress(ax, 1.38)
mark_enp_reform(ax)
ax.set_title('Seasonal_Factor — мультипликатор для агрегационного слоя LSI')
ax.set_ylabel('Множитель 1.0—1.4')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('График 3 — Фичи М4 во времени (красные пунктиры = unplanned-стресс)',
             fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / 'm4_g3_features_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print('Ключевое наблюдение для экспертов:')
print('  M4 не является самостоятельным стресс-сигналом: он показывает')
print('  календарно ожидаемую налоговую нагрузку вокруг каждой даты.')
print('  Если стресс совпал с tax_pressure, эту часть надо считать контекстом,')
print('  а непредсказуемый остаток оставлять в residual-сигналах M1/M2/M5.')


In [ ]:
# ============================================================
# ЯЧЕЙКА 12 — Подключение реального M1 и ручная STL-декомпозиция
#
# ТЗ по М4 требует показать пересечение налогового календаря с M1/M2/M5
# и предложить способ убрать двойной счёт. Здесь используем НЕ симуляцию,
# а готовый feature dataset модуля M1:
#   data/processed/m1_features.csv
#
# Берём реальные ряды:
#   spread             — запас/дефицит относительно усредняемых резервов
#   ruonia_period_avg  — средняя RUONIA за период усреднения
#
# STL-ДЕКОМПОЗИЦИЯ (ручная, без statsmodels):
#   trend    — скользящая медиана 13 месяцев (центрированная)
#   seasonal — среднее отклонение от тренда по календарному месяцу
#   resid    — остаток = ряд − trend − seasonal
#
# Идея для устранения двойного счёта:
#   налоговая сезонность остаётся в М4 как Seasonal_Factor,
#   а M1 для стресс-сигнала использует residual-компоненту.
# ============================================================

if not M1_FEATURES.exists():
    raise FileNotFoundError(f'Не найден датасет M1: {M1_FEATURES}')

m1_raw = pd.read_csv(M1_FEATURES)
m1_raw['period_start'] = pd.to_datetime(m1_raw['date'], format='%d-%m-%Y')
m1_raw['Дата'] = m1_raw['period_start'].dt.to_period('M').dt.to_timestamp()

# Оставляем период пересечения с M4. Это важно для честных корреляций.
m1 = m1_raw[(m1_raw['Дата'] >= df['date'].min().to_period('M').to_timestamp()) &
            (m1_raw['Дата'] <= df['date'].max().to_period('M').to_timestamp())].copy()

# Канонические имена, чтобы остальная аналитика была читаемой.
m1['месяц'] = m1['Дата'].dt.month
m1['год'] = m1['Дата'].dt.year
m1['спред'] = m1['spread'].astype(float)
m1['RUONIA'] = m1['ruonia_period_avg'].fillna(m1['ruonia_rate']).astype(float)

# Если в одном месяце окажется несколько периодов усреднения, агрегируем до месяца.
# Сейчас данных обычно одна строка на месяц, но группировка делает ноутбук устойчивее.
agg_spec = {
    'period_start': 'min',
    'averaging_period_end': 'last',
    'спред': 'mean',
    'RUONIA': 'mean',
    'spread_mad_score': 'mean',
    'ruonia_mad_score': 'mean',
    'm1_signal': 'mean',
    'm1_signal_final': 'mean',
    'm1_reliable': 'max',
}
m1 = m1.groupby('Дата', as_index=False).agg(agg_spec)
m1['месяц'] = m1['Дата'].dt.month
m1['год'] = m1['Дата'].dt.year

print(f'Загружен реальный M1 feature dataset: {len(m1):,} месячных наблюдений')
print(f'Период M1 в анализе: {m1.Дата.min().date()} -> {m1.Дата.max().date()}')
print(f'Покрытие RUONIA: {m1.RUONIA.notna().mean()*100:.1f}% наблюдений')

# === Manual STL ===
def manual_stl(series_indexed, period=12):
    s = series_indexed.astype(float).copy()
    # Trend: скользящая медиана 13 периодов, центрированная
    trend = s.rolling(period + 1, center=True, min_periods=max(3, period // 2)).median()
    trend = trend.bfill().ffill()

    # Seasonal: средняя по календарному месяцу
    detrended = s - trend
    months_idx = series_indexed.index.month
    seasonal = pd.Series(0.0, index=s.index)
    for m_num in range(1, period + 1):
        mask = months_idx == m_num
        seasonal.loc[mask] = detrended.loc[mask].mean(skipna=True)
    seasonal = seasonal.fillna(0.0)
    seasonal -= seasonal.mean()

    residual = s - trend - seasonal
    return trend, seasonal, residual

# Применяем STL к реальному спреду и RUONIA
m1_idx = m1.set_index('Дата')['спред']
trend_s, seasonal_s, resid_s = manual_stl(m1_idx, period=12)
m1['спред_trend'] = trend_s.values
m1['спред_seasonal'] = seasonal_s.values
m1['спред_resid'] = resid_s.values

ru_idx = m1.set_index('Дата')['RUONIA']
trend_r, seasonal_r, resid_r = manual_stl(ru_idx, period=12)
m1['RUONIA_trend'] = trend_r.values
m1['RUONIA_seasonal'] = seasonal_r.values
m1['RUONIA_resid'] = resid_r.values

# MAD-нормализация ДО и ПОСЛЕ декомпозиции.
# Raw пересчитываем тем же методом, чтобы сравнение raw vs residual было честным.
def mad_score_monthly(s, w=WINDOW):
    minp = 12
    rmed = s.rolling(w, min_periods=minp).median()
    rmad = s.rolling(w, min_periods=minp).apply(lambda x: median_abs_deviation(x, nan_policy='omit'), raw=False)
    floor = (rmed.abs() * 0.01).clip(lower=1e-6)
    rmad_safe = rmad.clip(lower=floor)
    return ((s - rmed) / rmad_safe).clip(-5, 5)

m1['MAD_спред_raw'] = mad_score_monthly(m1['спред'])
m1['MAD_спред_resid'] = mad_score_monthly(m1['спред_resid'])
m1['MAD_RUONIA_raw'] = mad_score_monthly(m1['RUONIA'])
m1['MAD_RUONIA_resid'] = mad_score_monthly(m1['RUONIA_resid'])

print()
print('STL-декомпозиция реального M1 готова.')
print('Сводка спреда M1 (последние 6 месяцев):')
display(m1[['Дата','спред','спред_trend','спред_seasonal','спред_resid',
            'MAD_спред_raw','MAD_спред_resid','m1_signal_final']].tail(6).round(2))


In [ ]:
# ============================================================
# ЯЧЕЙКА 13 — Корреляция M4 vs M1 — проверка пересечения
# и эффекта STL на реальных данных M1
#
# Ожидание: связь M4 с raw-компонентами M1 должна быть выше, чем
# связь M4 с residual M1 после удаления тренда и сезонности.
# Если residual всё ещё коррелирует с M4, значит нужна более сильная
# декомпозиция или условное взвешивание в агрегаторе.
# ============================================================

# Аггрегируем M4 до месяца для корреляции с M1
df['month_start'] = df['date'].values.astype('datetime64[M]')
m4_monthly = df.groupby('month_start').agg(
    tax_pressure_sum    =('tax_pressure', 'sum'),
    tax_pressure_mean   =('tax_pressure', 'mean'),
    tax_proximity_mean  =('tax_proximity','mean'),
    tax_payment_days    =('is_tax_payment_day','sum'),
    quarter_end_count   =('is_quarter_end','sum'),
    year_end_count      =('is_year_end','sum'),
    seasonal_factor_max =('Seasonal_Factor','max'),
    tax_events_total    =('tax_payment_events_count','sum'),
).reset_index().rename(columns={'month_start': 'Дата'})
m4_monthly['Дата'] = pd.to_datetime(m4_monthly['Дата'])

# Merge с M1
mrg = m1.merge(m4_monthly, on='Дата', how='left')

# Рассчитываем корреляции
m4_features = ['tax_pressure_sum', 'tax_pressure_mean', 'tax_proximity_mean',
               'tax_payment_days', 'quarter_end_count', 'year_end_count',
               'seasonal_factor_max', 'tax_events_total']
m1_raw = ['MAD_спред_raw', 'MAD_RUONIA_raw']
m1_resid = ['MAD_спред_resid', 'MAD_RUONIA_resid']

print('=' * 78)
print('  КОРРЕЛЯЦИЯ M4 ↔ M1 (raw)  vs  M4 ↔ M1 (residual после STL)')
print('=' * 78)
print()
print(f'{"M4 feature":<22}{"M1 col":<22}{"Pearson":>10}{"p-value":>10}{"Spearman":>10}')
print('-' * 78)

corr_table = []

for cm4 in m4_features:
    for cm1 in m1_raw + m1_resid:
        ser = mrg[[cm4, cm1]].dropna()
        if len(ser) < 30:
            continue
        r_p, p_p = pearsonr(ser[cm4], ser[cm1])
        r_s, _   = spearmanr(ser[cm4], ser[cm1])
        kind = 'raw' if cm1 in m1_raw else 'resid'
        corr_table.append({
            'm4': cm4, 'm1': cm1, 'kind': kind,
            'pearson': r_p, 'p': p_p, 'spearman': r_s
        })
        sig = '*' if p_p < 0.05 else ' '
        print(f'{cm4:<22}{cm1:<22}{r_p:>9.3f}{sig} {p_p:>9.3f}{r_s:>10.3f}')

corr_df = pd.DataFrame(corr_table)

print()
print('РЕЗЮМЕ:')
print('-' * 78)
raw_avg   = corr_df.loc[corr_df.kind=='raw',   'pearson'].abs().mean()
resid_avg = corr_df.loc[corr_df.kind=='resid', 'pearson'].abs().mean()
print(f'  Средний |Pearson| M4 vs raw M1:      {raw_avg:.3f}')
print(f'  Средний |Pearson| M4 vs residual M1: {resid_avg:.3f}')
reduction = (1 - resid_avg / max(raw_avg, 1e-9)) * 100
print(f'  Снижение корреляции: {reduction:.1f}%')
print()
print('Интерпретация:')
print('  Чем больше снижение — тем меньше риск двойного счёта между M4 и M1.')
print('  Если снижение слабое, это не провал M4: значит нужна доработка')
print('  декомпозиции или условное снижение веса M1/M2/M5 в Tax_Week.')


In [ ]:
# ============================================================
# ГРАФИК 4 — STL-декомпозиция спреда М1
# Trend / Seasonal / Residual — три компоненты
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(15, 11), sharex=True)

# 1. Original
ax = axes[0]
ax.plot(m1['Дата'], m1['спред'], color=C['blue'], linewidth=1.4)
ax.fill_between(m1['Дата'], 0, m1['спред'], color=C['blue'], alpha=0.15)
mark_stress(ax, m1['спред'].max() * 0.96)
ax.set_title('1. Исходный спред М1 (factory output)')
ax.set_ylabel('млрд руб.')
ax.yaxis.set_major_formatter(FMT_RUB)
ax.grid(True)

# 2. Trend
ax = axes[1]
ax.plot(m1['Дата'], m1['спред_trend'], color=C['gray'], linewidth=2)
ax.fill_between(m1['Дата'], 0, m1['спред_trend'], color=C['gray'], alpha=0.15)
mark_stress(ax, m1['спред_trend'].max() * 0.96)
ax.set_title('2. Trend — структурный уровень (rolling median 13 мес.)')
ax.set_ylabel('млрд руб.')
ax.yaxis.set_major_formatter(FMT_RUB)
ax.grid(True)

# 3. Seasonal
ax = axes[2]
ax.plot(m1['Дата'], m1['спред_seasonal'], color=C['purple'], linewidth=1.4)
ax.fill_between(m1['Дата'], 0, m1['спред_seasonal'],
                where=(m1['спред_seasonal']>0), color=C['purple'], alpha=0.30, label='Профицит сезонный')
ax.fill_between(m1['Дата'], 0, m1['спред_seasonal'],
                where=(m1['спред_seasonal']<0), color=C['teal'], alpha=0.30, label='Дефицит сезонный')
ax.axhline(0, color=C['gray'], linewidth=0.8)
mark_stress(ax, m1['спред_seasonal'].max() * 0.96)
ax.set_title('3. Seasonal — повторяющаяся годовая компонента (то что вычитаем для М4)')
ax.set_ylabel('млрд руб.')
ax.yaxis.set_major_formatter(FMT_RUB)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True)

# 4. Residual
ax = axes[3]
ax.plot(m1['Дата'], m1['спред_resid'], color=C['red'], linewidth=1.4)
ax.fill_between(m1['Дата'], 0, m1['спред_resid'],
                where=(m1['спред_resid']>0), color=C['red'], alpha=0.30, label='Аномалия выше нормы')
ax.fill_between(m1['Дата'], 0, m1['спред_resid'],
                where=(m1['спред_resid']<0), color=C['green'], alpha=0.30, label='Аномалия ниже нормы')
ax.axhline(0, color=C['gray'], linewidth=0.8)
mark_stress(ax, m1['спред_resid'].max() * 0.96)
ax.set_title('4. Residual — настоящие аномалии (на это идёт MAD в М1 после исправления)')
ax.set_ylabel('млрд руб.')
ax.yaxis.set_major_formatter(FMT_RUB)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('График 4 — STL-декомпозиция спреда М1\n'
             'Сезонная часть переходит в М4, MAD считается только от резидуала',
             fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / 'm4_g4_stl.png', dpi=150, bbox_inches='tight')
plt.show()

# Численная проверка
peak_dec14 = m1.loc[m1['Дата']=='2014-12-01', ['спред','спред_trend','спред_seasonal','спред_resid']].iloc[0]
print('Декомпозиция декабря 2014 (стресс-эпизод):')
print(f'  Исходный спред:  {peak_dec14["спред"]:>10,.1f} млрд')
print(f'  Trend:           {peak_dec14["спред_trend"]:>10,.1f} млрд')
print(f'  Seasonal:        {peak_dec14["спред_seasonal"]:>10,.1f} млрд (год-енд эффект)')
print(f'  Residual:        {peak_dec14["спред_resid"]:>10,.1f} млрд (это и есть кризис)')


In [ ]:
# ============================================================
# ГРАФИК 5 — Корреляционная матрица M4 vs M1
# ============================================================

# Подготавливаем матрицу
m4_cols_lbl = {
    'tax_pressure_sum':    'TaxPress sum',
    'tax_pressure_mean':   'TaxPress mean',
    'tax_proximity_mean':  'Proximity mean',
    'tax_payment_days':    'Payment days',
    'quarter_end_count':   'Q-end',
    'year_end_count':      'Y-end',
    'seasonal_factor_max': 'SeasFactor max',
}
m1_cols_lbl = {
    'MAD_спред_raw':    'MAD спред (raw)',
    'MAD_спред_resid':  'MAD спред (resid)',
    'MAD_RUONIA_raw':   'MAD RUONIA (raw)',
    'MAD_RUONIA_resid': 'MAD RUONIA (resid)',
}

corr_matrix = pd.DataFrame(
    index=list(m4_cols_lbl.values()),
    columns=list(m1_cols_lbl.values()),
    dtype=float
)

for cm4, lm4 in m4_cols_lbl.items():
    for cm1, lm1 in m1_cols_lbl.items():
        ser = mrg[[cm4, cm1]].dropna()
        if len(ser) >= 30:
            corr_matrix.loc[lm4, lm1] = pearsonr(ser[cm4], ser[cm1])[0]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(corr_matrix.values.astype(float),
               cmap='RdBu_r', vmin=-0.7, vmax=0.7, aspect='auto')

ax.set_xticks(range(len(m1_cols_lbl)))
ax.set_xticklabels(m1_cols_lbl.values(), rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(m4_cols_lbl)))
ax.set_yticklabels(m4_cols_lbl.values(), fontsize=10)

# Подписи в ячейках
for i in range(len(m4_cols_lbl)):
    for j in range(len(m1_cols_lbl)):
        v = corr_matrix.values[i, j]
        if pd.notna(v):
            color = 'white' if abs(v) > 0.35 else '#1a1d27'
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center',
                    color=color, fontsize=9, fontweight='bold')

# Делаем визуальное разделение между raw и resid колонками
ax.axvline(1.5, color=C['yellow'], linewidth=2)
ax.axvline(3.5, color=C['yellow'], linewidth=0)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Pearson correlation', color=C['gray'])

ax.set_title('График 5 — Корреляция М4 ↔ М1\n'
             'Слева от жёлтой линии — RAW M1, справа — RESIDUAL после STL.\n'
             'Колонки RESIDUAL должны быть ближе к 0 — это решает проблему двойного счёта',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'm4_g5_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('Численное резюме (среднее |corr|):')
raw_mean   = corr_matrix[['MAD спред (raw)', 'MAD RUONIA (raw)']].abs().mean().mean()
resid_mean = corr_matrix[['MAD спред (resid)', 'MAD RUONIA (resid)']].abs().mean().mean()
print(f'  M4 vs M1 raw:      {raw_mean:.3f}')
print(f'  M4 vs M1 resid:    {resid_mean:.3f}')
print(f'  Снижение:          {(1 - resid_mean/max(raw_mean,1e-9))*100:.1f}%')


In [ ]:
# ============================================================
# ГРАФИК 6 — Зум на 3 стресс-эпизода
#
# КЛЮЧЕВАЯ ДЕМОНСТРАЦИЯ для экспертов:
# В дни Дек 2014, Фев 2022, Авг 2023 — это unplanned стресс.
# M4 показывает календарно ожидаемую часть нагрузки,
# а M1_residual должен отражать непредсказуемую часть стресса.
# Важно: часть стресс-дней может совпадать с налоговыми датами,
# поэтому корректный вывод — не "М4 молчит", а "М4 отделяет календарный фон".
# ============================================================

episodes = [
    ('Декабрь 2014', '2014-09-01', '2015-03-01', '2014-12-16'),
    ('Февраль 2022', '2021-11-01', '2022-05-01', '2022-02-28'),
    ('Август 2023',  '2023-05-01', '2023-11-01', '2023-08-15'),
]

for title, start, end, crisis in episodes:
    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
    crisis_dt = pd.to_datetime(crisis)

    # M4 daily — tax_pressure
    s_m4 = df[(df['date']>=start) & (df['date']<=end)]
    ax = axes[0]
    ax.fill_between(s_m4['date'], 0, s_m4['tax_pressure_smoothed'],
                    color=C['orange'], alpha=0.4)
    ax.plot(s_m4['date'], s_m4['tax_pressure_smoothed'],
            color=C['orange'], linewidth=1.5, label='Tax pressure (smoothed)')
    ax.axvline(crisis_dt, color=C['red'], linestyle=':', linewidth=2)
    ax.text(crisis_dt, ax.get_ylim()[1]*0.92, ' ' + title,
            color=C['red'], fontweight='bold')
    ax.set_title(f'{title} — календарный фон М4 вокруг стресс-эпизода')
    ax.set_ylabel('Tax pressure')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True)

    # M1 monthly — raw spread vs residual
    s_m1 = m1[(m1['Дата']>=start) & (m1['Дата']<=end)]
    ax = axes[1]
    ax.plot(s_m1['Дата'], s_m1['спред'],
            color=C['blue'], marker='o', ms=6, label='Raw спред')
    ax.plot(s_m1['Дата'], s_m1['спред_seasonal'] + s_m1['спред_trend'],
            color=C['purple'], linestyle='--', marker='^', ms=4,
            label='Trend + Seasonal (предсказуемая часть)')
    ax.axvline(crisis_dt, color=C['red'], linestyle=':', linewidth=2)
    ax.set_title('Спред М1: что объясняется сезонностью, что — нет')
    ax.set_ylabel('млрд руб.')
    ax.yaxis.set_major_formatter(FMT_RUB)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True)

    # MAD — raw vs residual
    ax = axes[2]
    ax.plot(s_m1['Дата'], s_m1['MAD_спред_raw'],
            color=C['orange'], marker='o', ms=5, label='MAD спреда (raw)')
    ax.plot(s_m1['Дата'], s_m1['MAD_спред_resid'],
            color=C['red'], marker='s', ms=5, label='MAD спреда (residual)')
    ax.axhline(2.0, color=C['orange'], linestyle='--', alpha=0.6, label='Тревога')
    ax.axhline(3.0, color=C['red'], linestyle='--', alpha=0.6, label='Стресс')
    ax.axhline(0, color=C['gray'], linewidth=0.8)
    ax.axvline(crisis_dt, color=C['red'], linestyle=':', linewidth=2)
    ax.set_title('MAD-score: raw vs residual после удаления сезонной компоненты')
    ax.set_ylabel('MAD score')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    plt.suptitle(f'График 6.{episodes.index((title, start, end, crisis))+1} — '
                 f'{title}: календарный фон M4 и residual M1',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'm4_g6_zoom_{title.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Краткая статистика для эпизода
    crisis_date_m1 = crisis_dt.replace(day=1)
    row = m1.loc[m1['Дата']==crisis_date_m1]
    if not row.empty:
        row = row.iloc[0]
        print(f'{title}:')
        print(f'  Tax pressure:        {s_m4.loc[s_m4["date"]==crisis_dt, "tax_pressure"].values[0] if (s_m4["date"]==crisis_dt).any() else 0:.2f}'
              f' (база ~ {df.tax_pressure.mean():.2f})')
        print(f'  Seasonal_Factor:     {s_m4.loc[s_m4["date"]==crisis_dt, "Seasonal_Factor"].values[0] if (s_m4["date"]==crisis_dt).any() else 1.0:.2f}'
              f' (база 1.0)')
        print(f'  MAD спред raw:       {row["MAD_спред_raw"]:.2f}')
        print(f'  MAD спред residual:  {row["MAD_спред_resid"]:.2f}  (после удаления сезонности)')
        print()


In [ ]:
# ============================================================
# ГРАФИК 7 — Сезонность фич М4 по месяцам
# Какие месяцы — самые "налогово-нагруженные"?
# ============================================================

month_agg = df.groupby('месяц').agg(
    tax_pressure_mean=('tax_pressure','mean'),
    tax_proximity_mean=('tax_proximity','mean'),
    seasonal_factor_mean=('Seasonal_Factor','mean'),
    payment_days=('is_tax_payment_day','sum'),
).reset_index()

ML = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# tax_pressure по месяцам
overall = month_agg['tax_pressure_mean'].mean()
colors = [C['red'] if v > overall * 1.15 else C['blue'] for v in month_agg['tax_pressure_mean']]
ax1.bar(month_agg['месяц'], month_agg['tax_pressure_mean'],
        color=colors, alpha=0.85)
ax1.axhline(overall, color=C['orange'], linestyle='--',
            label=f'Среднее: {overall:.2f}')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(ML, fontsize=8)
ax1.set_title('Средний tax_pressure по месяцам')
ax1.set_ylabel('Tax pressure')
ax1.legend(fontsize=8)
ax1.grid(True, axis='y')

# Seasonal_Factor
ovr = month_agg['seasonal_factor_mean'].mean()
ax2.bar(month_agg['месяц'], month_agg['seasonal_factor_mean'],
        color=C['purple'], alpha=0.85)
ax2.axhline(ovr, color=C['orange'], linestyle='--',
            label=f'Среднее: {ovr:.3f}')
ax2.axhline(1.0, color=C['gray'], linestyle=':', alpha=0.5)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(ML, fontsize=8)
ax2.set_title('Средний Seasonal_Factor по месяцам')
ax2.set_ylabel('Множитель')
ax2.legend(fontsize=8)
ax2.grid(True, axis='y')
ax2.set_ylim(0.95, 1.30)

# Payment days
ax3.bar(month_agg['месяц'], month_agg['payment_days'],
        color=C['teal'], alpha=0.85)
ax3.set_xticks(range(1, 13))
ax3.set_xticklabels(ML, fontsize=8)
ax3.set_title('Число tax_payment_days за всю историю')
ax3.set_ylabel('Дней')
ax3.grid(True, axis='y')

plt.suptitle('График 7 — Сезонность М4: Mar/Apr/Jul/Oct/Dec — самые нагруженные',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'm4_g7_monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

print('Топ-5 месяцев по среднему tax_pressure:')
top = month_agg.nlargest(5, 'tax_pressure_mean')
for _, r in top.iterrows():
    print(f'  {ML[int(r.месяц)-1]}: {r.tax_pressure_mean:.3f}')


In [ ]:
# ============================================================
# ЯЧЕЙКА 18 — Проверка гипотезы и итоговый выход в агрегатор
# ============================================================

print('=' * 70)
print('  ГИПОТЕЗА МОДУЛЯ М4 И ЕЁ ПРОВЕРКА')
print('=' * 70)
print()
print('ГИПОТЕЗА:')
print('  Налоговый период создаёт ПРЕДСКАЗУЕМЫЙ всплеск активности на')
print('  денежном рынке. Без контекстуализации этот всплеск ошибочно')
print('  принимается за стресс. М4 — контекстный модуль:')
print('    1. Подсвечивает запланированный стресс через tax_pressure / Seasonal_Factor')
print('    2. На реальном M1 показывает STL-разделение сезонной и residual-части')
print('       как практический способ уменьшить двойной счёт в LSI.')
print('    3. На стресс-эпизодах М4 показывает календарно ожидаемую часть,')
print('       а residual M1 оставляет непредсказуемую часть стресса для LSI.')
print()
print('=' * 70)
print('  ВАЛИДАЦИЯ НА ИСТОРИЧЕСКИХ ЭПИЗОДАХ')
print('=' * 70)
print()
print(f'{"Эпизод":<14}{"Tax_Press":>12}{"SeasFactor":>12}{"MAD_raw":>10}{"MAD_resid":>12}{"|raw|":>10}{"|resid|":>10}')
print('-' * 70)

for name, date in STRESS.items():
    crisis_dt = pd.to_datetime(date)
    # M4 значения на дату
    m4_row = df.loc[df['date']==crisis_dt]
    if m4_row.empty:
        # ближайший
        m4_row = df.iloc[(df['date'] - crisis_dt).abs().argsort()[:1]]
    tp = m4_row['tax_pressure'].values[0] if not m4_row.empty else np.nan
    sf = m4_row['Seasonal_Factor'].values[0] if not m4_row.empty else np.nan

    # M1 значения на месяц
    m1_row = m1.loc[m1['Дата'] == crisis_dt.replace(day=1)]
    raw   = m1_row['MAD_спред_raw'].values[0]   if not m1_row.empty else np.nan
    resid = m1_row['MAD_спред_resid'].values[0] if not m1_row.empty else np.nan

    print(f'{name:<14}{tp:>12.2f}{sf:>12.2f}{raw:>10.2f}{resid:>12.2f}{abs(raw):>10.2f}{abs(resid):>10.2f}')

print()
print('Интерпретация: таблица показывает, совпадает ли стресс-дата с налоговым фоном')
print('и насколько raw/residual M1 отличаются после удаления сезонной компоненты.')
print('Если residual остаётся высоким, это непредсказуемая часть стресса;')
print('если residual падает относительно raw, значит часть сигнала была календарной.')
print()
print('=' * 70)
print('  ВЫХОД М4 → АГРЕГАЦИОННЫЙ СЛОЙ (LSI)')
print('=' * 70)
print()

output_cols = [
    'date',
    # Бинарные флаги — идут в агрегатор как есть
    'Tax_Pre_Flag', 'Tax_Active_Flag', 'Tax_Post_Flag',
    'Tax_Week_Flag', 'Tax_Day_Strict',
    'is_quarter_end', 'is_year_end', 'is_month_end',
    'Regime_Post_ENP',
    # Непрерывные фичи
    'tax_pressure', 'tax_pressure_smoothed',
    'tax_proximity',
    # MAD-нормализованные
    'MAD_tax_pressure', 'MAD_tax_proximity',
    # Главный выход — мультипликатор
    'Seasonal_Factor', 'Seasonal_Factor_raw',
]

m4_export = df[output_cols].copy()
m4_export.to_csv(M4_EXPORT_CSV, index=False)
try:
    m4_export.to_parquet(M4_EXPORT_PARQUET, index=False)
    parquet_msg = f' и {M4_EXPORT_PARQUET.relative_to(PROJECT_ROOT)}'
except Exception as exc:
    parquet_msg = f' (parquet не сохранён: {exc})'

print(f'Сохранено в {M4_EXPORT_CSV.relative_to(PROJECT_ROOT)}{parquet_msg}: {len(m4_export):,} строк × {len(output_cols)} колонок')
print()
print('Что идёт в агрегационный слой LSI:')
print('  Бинарные флаги (для XGBoost / SHAP):')
print('    Tax_Pre_Flag, Tax_Active_Flag, Tax_Post_Flag')
print('    Tax_Week_Flag, Tax_Day_Strict')
print('    is_quarter_end, is_year_end, is_month_end')
print('    Regime_Post_ENP')
print('  Непрерывные:')
print('    tax_pressure, tax_pressure_smoothed, tax_proximity')
print('    MAD_tax_pressure, MAD_tax_proximity')
print('  Главный выход:')
print('    Seasonal_Factor (1.0–1.4) — мультипликатор LSI')
print()
print('Последние 5 строк выгрузки:')
display(m4_export.tail(5))

print()
print('=' * 70)
print('  МЕТОДОЛОГИЧЕСКАЯ ЗАМЕТКА (для экспертов)')
print('=' * 70)
print('''
ПРОБЛЕМА ДВОЙНОГО СЧЁТА — РЕШЕНИЕ:
  ТЗ требует устранить пересечение M4 с M1/M2/M5.
  В этом ноутбуке подход через STL-декомпозицию численно показан
  на реальном feature dataset модуля M1. Для M2/M5 применяется
  тот же принцип на их исходных рядах перед агрегацией.

  1. Исходный ряд модуля раскладывается на trend+seasonal+resid.
  2. MAD-нормализация делается ТОЛЬКО на residual.
  3. Сезонная компонента полностью переходит в М4 как Seasonal_Factor.
  4. На уровне агрегатора LSI = f(MAD-resids всех модулей) × Seasonal_Factor.

ПРЕИМУЩЕСТВА ПОДХОДА:
  • Каждый модуль вносит уникальный сигнал — нет дублирования.
  • SHAP-разложение на этапе агрегатора будет интерпретируемым:
    "В декабре 2014 LSI=85: вклад MAD-resid M1 = +12, M2 = +8, ...
     Seasonal_Factor = 1.20 — закладываем сезонную ожидаемость."
  • На стресс-эпизодах Seasonal_Factor показывает календарный фон,
    а LSI должен расти за счёт residual-компонент там, где есть
    непредсказуемая часть стресса.

ОГРАНИЧЕНИЯ:
  • Ручная STL не использует LOESS — для production стоит подключить
    statsmodels.tsa.seasonal.STL для более точной декомпозиции.
  • Реформа ЕНП 2023 создаёт структурный разрыв — рекомендуется
    делать STL отдельно по pre/post-режимам или включить regime dummy
    в residual model.
  • Праздничные сдвиги уплат (28-е попало на воскресенье) уже
    разрешены в датасете, но если в будущем источник изменится —
    нужно проверять явно через is_weekend × is_tax_payment_day.
''')
